In [ ]:
import math
import json
import requests
import pandas as pd
from tqdm import tqdm

url = "https://tenup.fft.fr/back/public/v2/classements/recherche"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://tenup.fft.fr",
    "Referer": "https://tenup.fft.fr/classement-padel?categorie=H",
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}

cookies = {
    "datadome": "yqabVQyu9b0E9h_bXOHh4bV0mUXEhjorSk36BR8DMg1ZyxbxWGzZZUFtK_WMOwLzxMyoR_NEE1~IAluVgOzqJZ6T3TrUarz6iIzGSGh6L4z6aweXT6B_nUIhSnSPR_84",
}

taille_page = 100  # mets une valeur réaliste
payload = {
    "categorie": "H",
    "sexe": "H",
    "pratique": "PADEL",
    "page": 1,
    "taillePage": taille_page
}

session = requests.Session()

# 1) Première requête pour récupérer total
response = session.post(
    url,
    headers=headers,
    cookies=cookies,
    json=payload,
    timeout=30
)

response.raise_for_status()
data = response.json()

total = data["total"]
nb_pages = math.ceil(total / taille_page)

print(f"Total joueurs : {total}")
print(f"Taille page   : {taille_page}")
print(f"Nb pages      : {nb_pages}")

# 2) On stocke déjà la première page
all_dfs = []

if "joueurs" in data and data["joueurs"]:
    df_page_1 = pd.json_normalize(data["joueurs"])
    df_page_1["page_source"] = 1
    all_dfs.append(df_page_1)

# 3) Boucle sur les pages restantes
for page in tqdm(range(2, nb_pages + 1), desc="Récupération des pages"):
    payload["page"] = page

    response = session.post(
        url,
        headers=headers,
        cookies=cookies,
        json=payload,
        timeout=30
    )

    try:
        response.raise_for_status()
        data = response.json()

        joueurs = data.get("joueurs", [])
        if joueurs:
            df_page = pd.json_normalize(joueurs)
            df_page["page_source"] = page
            all_dfs.append(df_page)

    except Exception as e:
        print(f"Erreur page {page}: {e}")
        print(response.text[:1000])

# 4) Concat final
if all_dfs:
    df_final = pd.concat(all_dfs, ignore_index=True)
else:
    df_final = pd.DataFrame()

print(df_final.shape)
print(df_final.head())

# 5) Optionnel : suppression des doublons éventuels
if "idCrm" in df_final.columns:
    df_final = df_final.drop_duplicates(subset=["idCrm"], keep="first")

print("Shape après dédoublonnage :", df_final.shape)

# 6) Export
df_final.to_csv("joueurs_padel.csv", index=False, encoding="utf-8-sig")
print("CSV exporté : joueurs_padel.csv")

Total joueurs : 141899
Taille page   : 100
Nb pages      : 1419


Récupération des pages: 100%|██████████| 1418/1418 [06:07<00:00,  3.86it/s]
C:\Users\gwen\AppData\Local\Temp\ipykernel_12372\3614513684.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(all_dfs, ignore_index=True)


(141898, 19)
   position        idCrm                           club  classement  \
0         1   1329333964           AS PADEL AFICIONADOS           1   
1         2   9271826837    CHARTRES BRETAGNE ESPERANCE           1   
2         3  10006106112      ALL IN PADEL SPORTS ARENA           1   
3         4  10007146252  PADEL TOUCH BASSIN D'ARCACHON           1   
4         5  10007163685      ALL IN PADEL SPORTS ARENA           1   

   classementFip  evolution               nom   prenom  meilleurClassement  \
0          157.0        0.0            LEYGUE   Thomas                 1.0   
1          103.0        0.0          GUICHARD    Dylan                 1.0   
2            NaN        NaN              LIJO   Pablo                  1.0   
3            NaN        NaN  JOFRE MANZANARES    Inigo                 1.0   
4            NaN        NaN             RUBIO  Gonzalo                 1.0   

  nationalite  ageSportif  points  nombreTournoisJoues  \
0         FRA        25.0     NaN

: 